# FormoSpeech Whisper-v3 — Colab 手動驗證

## 用途

本 notebook **僅供手動 Colab 驗證**，用於確認 `formospeech/whisper-large-v3-taiwanese-hakka`
模型對合成/合法公開授權客語音檔的辨識能力。

**本 notebook 不執行 Production Invocation、不呼叫 AWS adapter、不連接任何實際 AWS 服務/SDK/network。**

| 欄位 | 值 |
|------|-----|
| Model ID | `formospeech/whisper-large-v3-taiwanese-hakka` |
| 授權 | CC BY-NC 4.0 |
| 存取方式 | Gated Model（需 HuggingFace token） |
| 用途限制 | `colab_validation_only` |

### 前置條件

1. 選擇免費 GPU runtime（Runtime → Change runtime type → GPU）
2. 已在 Colab Secrets 設定 `HF_TOKEN`
3. 已申請且取得 gated-model 存取權限

---
## Cell 1: GPU Preflight Check

In [ ]:
"""GPU 前置條件檢查：偵測 GPU 可用性，無 GPU 則輸出失敗分類與 retry step。"""
import json
import torch

if not torch.cuda.is_available():
    failure = {
        "failure_prerequisite": "GPU runtime not available",
        "failure_category": "gpu_unavailable",
        "retry_step": "請至 Runtime → Change runtime type → 選擇 GPU（T4），然後重新執行此 cell。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("GPU preflight failed.")

print(f"✓ GPU 可用：{torch.cuda.get_device_name(0)}")
print(f"  CUDA 版本：{torch.version.cuda}")
print(f"  可用記憶體：{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Cell 2: 精確依賴安裝

In [ ]:
"""從 requirements.lock 安裝精確版本依賴。"""
import subprocess
import sys

REQUIREMENTS = [
    "torch==2.1.2",
    "transformers==4.36.2",
    "accelerate==0.25.0",
    "soundfile==0.12.1",
    "librosa==0.10.1",
    "pydub==0.25.1",
    "numpy==1.26.2",
]

try:
    for pkg in REQUIREMENTS:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg]
        )
    print("✓ 所有依賴安裝完成（精確版本）")
except subprocess.CalledProcessError as e:
    failure = {
        "failure_prerequisite": f"Dependency installation failed: {e.cmd}",
        "failure_category": "dependency_install_failure",
        "retry_step": "請檢查網路連線，或嘗試重新啟動 runtime 後再次執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("Dependency install failed.")

---
## Cell 3: HF Token 讀取（僅 Colab Secret / runtime env）

In [ ]:
"""僅從 Colab Secret 或短生命週期 runtime environment 讀取 HF Token。

Token 不得寫入 notebook source、cell output、requirements.lock、evidence 或 ADR。
"""
import os

hf_token = None

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    failure = {
        "failure_prerequisite": "HF_TOKEN not found in Colab Secret or runtime environment",
        "failure_category": "token_missing",
        "retry_step": "請在 Colab 左側面板 → Secrets → 新增 HF_TOKEN，值為你的 HuggingFace token，然後重新執行此 cell。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("HF Token missing.")

# 驗證 token 格式最低要求（非空白字串）
if not isinstance(hf_token, str) or not hf_token.strip():
    failure = {
        "failure_prerequisite": "HF_TOKEN is blank or invalid format",
        "failure_category": "token_invalid",
        "retry_step": "請確認 Colab Secret 中 HF_TOKEN 的值為有效的 HuggingFace Access Token。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("HF Token invalid.")

print("✓ HF Token 已從安全來源讀取（不顯示內容）")

---
## Cell 4: Gated-Model 存取權限檢查

In [ ]:
"""確認 HF Token 具有 formospeech/whisper-large-v3-taiwanese-hakka 的 gated-model 存取權限。"""
from huggingface_hub import HfApi

MODEL_ID = "formospeech/whisper-large-v3-taiwanese-hakka"

try:
    api = HfApi(token=hf_token)
    model_info = api.model_info(MODEL_ID)
    print(f"✓ Gated-model 存取確認：{MODEL_ID}")
    print(f"  Model revision: {model_info.sha}")
except Exception as e:
    error_msg = str(e)
    if "401" in error_msg or "403" in error_msg or "gated" in error_msg.lower():
        failure = {
            "failure_prerequisite": "Gated-model access not granted for formospeech/whisper-large-v3-taiwanese-hakka",
            "failure_category": "gated_access_denied",
            "retry_step": "請至 https://huggingface.co/formospeech/whisper-large-v3-taiwanese-hakka 申請存取權限，等待核准後重新執行。"
        }
    else:
        failure = {
            "failure_prerequisite": f"Model access check failed: {type(e).__name__}",
            "failure_category": "model_access_check_failure",
            "retry_step": "請確認網路連線正常，並確認 HF_TOKEN 有效後重新執行。"
        }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("Gated-model access check failed.")

---
## Cell 5: 模型下載

In [ ]:
"""下載 formospeech/whisper-large-v3-taiwanese-hakka（使用 HF Token）。"""
from transformers import WhisperProcessor, WhisperForConditionalGeneration

try:
    processor = WhisperProcessor.from_pretrained(MODEL_ID, token=hf_token)
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, token=hf_token
    ).to("cuda")
    print(f"✓ 模型下載完成：{MODEL_ID}")
    print(f"  Device: {next(model.parameters()).device}")
except Exception as e:
    failure = {
        "failure_prerequisite": f"Model download failed: {type(e).__name__}",
        "failure_category": "model_download_failure",
        "retry_step": "請確認網路連線穩定、GPU 記憶體足夠（建議 T4 以上），並確認 gated-model 存取已核准後重新執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("Model download failed.")

---
## Cell 6: Formo Prompt ID 驗證

In [ ]:
"""Formo Prompt ID 精確 allowlist 驗證。

僅接受精確匹配六個允許值；拒絕空白、大小寫變形、前後空白與 Unicode lookalike。
不做任何正規化或猜測。使用 task 2 的 exact validator 邏輯。
"""

FORMO_PROMPT_ALLOWLIST = frozenset({
    "htia_sixian", "htia_hailu", "htia_dapu",
    "htia_raoping", "htia_zhaoan", "htia_nansixian"
})


def validate_formo_prompt_id(candidate: str) -> bool:
    """僅接受精確匹配六個允許值；拒絕空白、大小寫變形、前後空白與 Unicode lookalike。"""
    return candidate in FORMO_PROMPT_ALLOWLIST


# 設定要驗證的 Prompt ID（依據欲辨識的客語腔調選擇）
PROMPT_ID = "htia_sixian"  # 可改為其他五個允許值

if not validate_formo_prompt_id(PROMPT_ID):
    failure = {
        "failure_prerequisite": f"Invalid Formo Prompt ID: {PROMPT_ID!r}",
        "failure_category": "formo_prompt_id_invalid",
        "retry_step": f"Prompt ID 必須精確為以下之一：{sorted(FORMO_PROMPT_ALLOWLIST)}。請修正後重新執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("Formo Prompt ID validation failed.")

print(f"✓ Formo Prompt ID 驗證通過：{PROMPT_ID}")

---
## Cell 7: WAV 輸入處理與推論

In [ ]:
"""載入 WAV fixture 並執行推論。

僅接受 fixture_provenance.json 中宣告的合成/合法公開授權音檔。
"""
import soundfile as sf
import librosa
import numpy as np

# WAV fixture 路徑（執行前請上傳合成測試音檔）
WAV_FIXTURE_PATH = "test_fixture.wav"  # 請替換為實際上傳的 fixture 檔名
WAV_FIXTURE_ID = "formo-synth-wav-001"

try:
    # 載入音訊並轉為 16kHz mono
    audio_array, sr = librosa.load(WAV_FIXTURE_PATH, sr=16000, mono=True)
    audio_duration_ms = int(len(audio_array) / sr * 1000)
    print(f"✓ WAV 載入成功：{audio_duration_ms} ms, sample rate={sr}")
except FileNotFoundError:
    failure = {
        "failure_prerequisite": f"WAV fixture not found: {WAV_FIXTURE_PATH}",
        "failure_category": "wav_input_missing",
        "retry_step": "請上傳合成測試 WAV 檔案至 Colab runtime，並確認檔名正確後重新執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("WAV fixture not found.")
except Exception as e:
    failure = {
        "failure_prerequisite": f"WAV decode failed: {type(e).__name__}",
        "failure_category": "wav_decode_failure",
        "retry_step": "請確認 WAV 檔案未損壞且為有效的 WAV 格式後重新執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("WAV decode failed.")

# 執行推論（使用已驗證的 Prompt ID）
try:
    input_features = processor(
        audio_array, sampling_rate=16000, return_tensors="pt"
    ).input_features.to("cuda")

    # 以 Prompt ID 建立 forced decoder IDs
    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="zh", task="transcribe"
    )

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=448
        )

    transcription = processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]
    wav_transcript_char_count = len(transcription)
    wav_success = wav_transcript_char_count > 0
    print(f"✓ WAV 推論完成：transcript_character_count={wav_transcript_char_count}")
    # 注意：不顯示完整逐字稿內容
except Exception as e:
    failure = {
        "failure_prerequisite": f"WAV inference failed: {type(e).__name__}",
        "failure_category": "inference_failure",
        "retry_step": "請確認 GPU 記憶體充足、模型已正確載入後重新執行。若記憶體不足請嘗試較短的音檔。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    raise SystemExit("WAV inference failed.")

---
## Cell 8: M4A 解碼處理

In [ ]:
"""M4A 解碼為 WAV 並執行推論。

任何 M4A decode failure 輸出 failure_prerequisite、failure_category 與 retry step。
"""
from pydub import AudioSegment
import tempfile

# M4A fixture 路徑（執行前請上傳合成測試音檔）
M4A_FIXTURE_PATH = "test_fixture.m4a"  # 請替換為實際上傳的 fixture 檔名
M4A_FIXTURE_ID = "formo-synth-m4a-001"

m4a_success = False
m4a_transcript_char_count = 0
m4a_audio_duration_ms = 0

try:
    # M4A 解碼
    audio_segment = AudioSegment.from_file(M4A_FIXTURE_PATH, format="m4a")
    m4a_audio_duration_ms = len(audio_segment)

    # 轉為 WAV 暫存檔
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        audio_segment.export(tmp.name, format="wav")
        tmp_wav_path = tmp.name

    # 載入解碼後的 WAV
    m4a_audio_array, m4a_sr = librosa.load(tmp_wav_path, sr=16000, mono=True)
    print(f"✓ M4A 解碼成功：{m4a_audio_duration_ms} ms")

    # 執行推論
    m4a_input_features = processor(
        m4a_audio_array, sampling_rate=16000, return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        m4a_predicted_ids = model.generate(
            m4a_input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=448
        )

    m4a_transcription = processor.batch_decode(
        m4a_predicted_ids, skip_special_tokens=True
    )[0]
    m4a_transcript_char_count = len(m4a_transcription)
    m4a_success = m4a_transcript_char_count > 0
    print(f"✓ M4A 推論完成：transcript_character_count={m4a_transcript_char_count}")

except FileNotFoundError:
    failure = {
        "failure_prerequisite": f"M4A fixture not found: {M4A_FIXTURE_PATH}",
        "failure_category": "m4a_input_missing",
        "retry_step": "請上傳合成測試 M4A 檔案至 Colab runtime，並確認檔名正確後重新執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    print("⚠ M4A 流程跳過，僅產生 WAV evidence。")
except Exception as e:
    failure = {
        "failure_prerequisite": f"M4A decode/inference failed: {type(e).__name__}",
        "failure_category": "m4a_decode_failure",
        "retry_step": "請確認 M4A 檔案未損壞、ffmpeg 可用（pydub 需要），並確認格式為有效 M4A/AAC 後重新執行。"
    }
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    print("⚠ M4A 流程失敗，僅產生 WAV evidence。")

---
## Cell 9: 結構化 Evidence 輸出

In [ ]:
"""產生去識別化的結構化 evidence record。

禁止欄位：transcript, token, hf_token, audio, audio_bytes, pcm_samples,
          prompt_id, formo_prompt_id, raw_response, raw_provider_response
"""
import hashlib
import uuid
from datetime import datetime, timezone

# 計算 requirements.lock 的 SHA-256 digest
REQUIREMENTS_LOCK_CONTENT = """torch==2.1.2
transformers==4.36.2
accelerate==0.25.0
soundfile==0.12.1
librosa==0.10.1
pydub==0.25.1
numpy==1.26.2
"""
manifest_digest = hashlib.sha256(REQUIREMENTS_LOCK_CONTENT.encode("utf-8")).hexdigest()

# 取得 model revision
try:
    model_revision = model_info.sha
except Exception:
    model_revision = "unknown"

# WAV evidence record
wav_evidence = {
    "schema_version": "1.0.0",
    "run_id": str(uuid.uuid4()),
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "model_id": "formospeech/whisper-large-v3-taiwanese-hakka",
    "model_revision": model_revision,
    "language": "hak",
    "input_format": "wav",
    "input_fixture_id": WAV_FIXTURE_ID,
    "audio_duration_ms": audio_duration_ms,
    "runtime_kind": "colab_free_gpu",
    "dependency_manifest_digest": manifest_digest,
    "outcome": "success" if wav_success else "failure",
    "failure_prerequisite": "" if wav_success else "Inference produced empty transcript",
    "failure_category": "" if wav_success else "inference_empty_output",
    "transcript_present": wav_success,
    "transcript_character_count": wav_transcript_char_count if wav_success else 0,
    "evidence_redaction_version": "1.0.0"
}

print("=== WAV Evidence Record ===")
print(json.dumps(wav_evidence, ensure_ascii=False, indent=2))

# M4A evidence record（如果有執行）
if m4a_audio_duration_ms > 0:
    m4a_evidence = {
        "schema_version": "1.0.0",
        "run_id": str(uuid.uuid4()),
        "recorded_at": datetime.now(timezone.utc).isoformat(),
        "model_id": "formospeech/whisper-large-v3-taiwanese-hakka",
        "model_revision": model_revision,
        "language": "hak",
        "input_format": "m4a",
        "input_fixture_id": M4A_FIXTURE_ID,
        "audio_duration_ms": m4a_audio_duration_ms,
        "runtime_kind": "colab_free_gpu",
        "dependency_manifest_digest": manifest_digest,
        "outcome": "success" if m4a_success else "failure",
        "failure_prerequisite": "" if m4a_success else "M4A inference produced empty transcript",
        "failure_category": "" if m4a_success else "inference_empty_output",
        "transcript_present": m4a_success,
        "transcript_character_count": m4a_transcript_char_count if m4a_success else 0,
        "evidence_redaction_version": "1.0.0"
    }
    print("\n=== M4A Evidence Record ===")
    print(json.dumps(m4a_evidence, ensure_ascii=False, indent=2))

print("\n✓ Evidence 輸出完成。不含 transcript、token、audio 或 prompt_id。")